In [ ]:
# -*- coding: utf-8 -*-

Qwen3_(4B)-GRPO.ipynb
#
Automatically generated by Colab.
#
Original file is located at
    https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb
#
To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>
#
To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
#
You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it
#
### News
#
Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)
#
<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>
#
Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)
#
Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)
#
New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)
#
Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).
#
### Installation

In [ ]:

# === Offline Kaggle wheels install (RTX PRO 6000 Blackwell / Nemotron challenge) ===
# Kaggle sessions run with the internet DISABLED, so we install from a public Kaggle
# dataset of pre-compiled wheels. Recommended dataset (most battle-tested, referenced
# by actual Nemotron challenge solutions):
#     mayukh18/nemotron-packages
# Fallbacks (same wheel content, different uploaders / newer TRL wheels):
#     llkh0a/rtx-wheels
#     farmountain/nemotron-trl-wheels     (updated 2026-07, TRL-focused)
#     dennisfong/nvidia-nemotron-offline-packages
# Install pattern (offline):
#     pip install --no-index --find-links=/kaggle/input/datasets/mayukh18/nemotron-packages/packages
# NOTE: to pin exact versions (trl==0.22.2, peft==0.18.0, transformers==4.56.2) the wheels
# dataset must contain them; the Kaggle Dependency Manager (Add-ons) is the alternative.
# To run this on a machine WITH internet, just run:  pip install unsloth vllm
import os, subprocess, sys


def _kaggle_offline_install():
    """Try offline wheels from Kaggle input datasets; fall back to online pip."""
    wheel_datasets = [
        # (dataset dir, subfolder containing wheels)
        ("datasets/mayukh18/nemotron-packages", "packages"),
        ("datasets/llkh0a/rtx-wheels", "wheels"),
        ("datasets/farmountain/nemotron-trl-wheels", "packages"),
        ("datasets/dennisfong/nvidia-nemotron-offline-packages", ""),
        ("datasets/ziechan/unsloth-for-offline", ""),
        ("datasets/nguyentuoc/unsloth-package-offline", ""),
        ("datasets/dungxnd/unsloth", ""),
        ("datasets/ethanyee2706/unsloth-wheels", ""),
        ("mayukh18/nemotron-packages", "packages"),
        ("llkh0a/rtx-wheels", "packages"),
        ("farmountain/nemotron-trl-wheels", "packages"),
        ("nvidia-nemotron-offline-packages", ""),
        ("unsloth-for-offline", ""),
        ("nemotron-packages", "packages"),
    ]
    for ds, sub in wheel_datasets:
        link = os.path.join("/kaggle/input", ds, sub)
        if os.path.isdir(link) and any(
            f.endswith(".whl") for _, _, files in os.walk(link) for f in files
        ):
            print(f"[install] Using offline wheels from: /kaggle/input/{ds}/{sub}")
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--no-index",
                    f"--find-links={link}",
                    "unsloth",
                    "vllm",
                    "bitsandbytes",
                    "xformers",
                ]
            )
            # Prefer exact versions known to work with unsloth 2026.8.x; allow them to be absent.
            for pkg in ["trl==0.22.2", "peft==0.18.0", "transformers==4.56.2"]:
                try:
                    subprocess.check_call(
                        [
                            sys.executable,
                            "-m",
                            "pip",
                            "install",
                            "-q",
                            "--no-index",
                            f"--find-links={link}",
                            pkg,
                        ]
                    )
                except subprocess.CalledProcessError:
                    print(
                        f"[install] Wheel {pkg} not in dataset — keeping preinstalled version."
                    )
            return
    print("[install] No offline Kaggle wheels found — attempting online pip install...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "unsloth",
            "vllm",
            "bitsandbytes",
            "xformers",
            "trl==0.22.2",
            "peft==0.18.0",
            "transformers==4.56.2",
        ]
    )


if os.path.isdir("/kaggle/input") and not os.getenv("UNSLOTH_SKIP_INSTALL"):
    try:
        _kaggle_offline_install()
    except Exception as e:
        print(
            f"[install] Wheel install failed ({e}) — continuing; imports may fail if unsloth is missing."
        )
os.environ.setdefault("UNSLOTH_VLLM_STANDBY", "1")  # Extra 30% context lengths

### Unsloth
#
Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.
#
We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [ ]:

# === Ablation / experiment configuration (CLI) ===
import argparse
import json
import pathlib
import sys


def parse_args(argv=None):
    p = argparse.ArgumentParser(
        description="Qwen3-4B GRPO math-reasoning fine-tuning (ablation-ready, WandB-tracked).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    # --- Model / data paths (offline: point to /kaggle/input copies) ---
    p.add_argument(
        "--model-path",
        type=str,
        default="unsloth/Qwen3-4B-Base",
        help="HF model id or local path (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--grpo-data-path",
        type=str,
        default="open-r1/DAPO-Math-17k-Processed",
        help="GRPO training dataset (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--sft-data-path",
        type=str,
        default="unsloth/OpenMathReasoning-mini",
        help="Pre-SFT formatting dataset (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--sample-n",
        type=int,
        default=0,
        help="Subset to first N GRPO samples (0 = use all)",
    )
    p.add_argument(
        "--max-seq-length",
        type=int,
        default=2048,
        help="Max sequence length (context window)",
    )
    p.add_argument("--sft-epochs", type=int, default=2, help="Pre-SFT epochs")
    p.add_argument(
        "--no-sft",
        action="store_true",
        help="Skip pre-SFT formatting (GRPO-from-base comparison arm)",
    )

    # --- GRPO ablation knobs (each is an experimental axis) ---
    p.add_argument(
        "--beta",
        type=float,
        default=0.001,
        help="KL-divergence penalty coefficient (0 disables KL & its logging). "
        "Note: trl only logs `kl` when beta != 0.",
    )
    p.add_argument(
        "--num-generations",
        type=int,
        default=4,
        choices=[2, 4, 8, 16],
        help="Group size: completions sampled per prompt (num_generations)",
    )
    p.add_argument(
        "--importance-sampling-level",
        type=str,
        default="token",
        choices=["token", "sequence"],
        help="Advantage credit assignment granularity (token-level vs sequence-level)",
    )
    p.add_argument(
        "--loss-type",
        type=str,
        default="grpo",
        choices=["grpo", "bnpo", "dr_grpo", "dapo"],
        help="GRPO loss variant (normalization differences)",
    )
    p.add_argument(
        "--temperature", type=float, default=1.0, help="Sampling temperature"
    )
    p.add_argument("--lr", type=float, default=5e-6, help="Learning rate")
    p.add_argument(
        "--optim",
        type=str,
        default="adamw",
        choices=["adamw", "adamw_8bit"],
        help="Optimizer (adamw default: bitsandbytes has a shape bug on Blackwell)",
    )
    p.add_argument("--max-steps", type=int, default=100, help="Training steps")
    p.add_argument("--seed", type=int, default=3407, help="Global seed")

    # --- Run identity / tracking ---
    p.add_argument(
        "--run-name",
        type=str,
        default=None,
        help="Run id (auto-generated from ablation axes if omitted)",
    )
    p.add_argument(
        "--wandb",
        dest="wandb",
        action="store_true",
        default=True,
        help="Enable WandB logging",
    )
    p.add_argument(
        "--no-wandb", dest="wandb", action="store_false", help="Disable WandB logging"
    )
    p.add_argument(
        "--wandb-project",
        type=str,
        default="grpo-math-ablation",
        help="WandB project (env WANDB_PROJECT overrides)",
    )
    p.add_argument(
        "--wandb-group",
        type=str,
        default="default",
        help="WandB run group (use ablation axis, e.g. beta, num_generations)",
    )
    p.add_argument(
        "--offline-mode",
        action="store_true",
        help="Force WandB offline (no network; sync later with `wandb sync`)",
    )
    return p.parse_args(argv)


args = parse_args()
RUN_NAME = args.run_name or "_".join(
    [
        f"beta{args.beta}",
        f"ng{args.num_generations}",
        f"loss_{args.loss_type}",
        f"is_{args.importance_sampling_level}",
        f"temp{args.temperature}",
        f"lr{args.lr}",
        f"opt{args.optim}",
    ]
    + (["noSFT"] if args.no_sft else [])
    + ([f"n{args.sample_n}"] if args.sample_n else [])
)
output_dir = pathlib.Path("outputs") / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

# --- WandB env (API key read from environment automatically) ---
if args.wandb:
    os.environ.setdefault("WANDB_API_KEY", os.environ.get("WANDB_API_KEY", ""))
    os.environ.setdefault("WANDB_PROJECT", args.wandb_project)
    os.environ.setdefault("WANDB_RUN_GROUP", args.wandb_group)
    os.environ.setdefault("WANDB_NAME", RUN_NAME)
    if args.offline_mode:
        os.environ["WANDB_MODE"] = "offline"
        print(
            "[wandb] Offline mode enabled (WANDB_MODE=offline) — sync later with `wandb sync`"
        )
    if not os.environ.get("WANDB_API_KEY"):
        print(
            "[wandb] WANDB_API_KEY not found in environment — falling back to offline mode "
            "(run dir saved under ./wandb; sync with `wandb sync` when online)"
        )
        os.environ["WANDB_MODE"] = "offline"

from unsloth import FastLanguageModel
import torch

max_seq_length = args.max_seq_length  # Can increase for longer reasoning traces
lora_rank = 32  # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=args.model_path,
    max_seq_length=max_seq_length,
    load_in_4bit=False,  # False for LoRA 16bit
    fast_inference=True,  # Enable vllm fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,  # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=lora_rank * 2,  # *2 speeds up training
    use_gradient_checkpointing="unsloth",  # Reduces memory usage
    random_state=args.seed,
)

### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:

reasoning_start = "<start_working_out>"  # Acts as think-open tag
reasoning_end = "<end_working_out>"  # Acts as think-close tag
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [ ]:

chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
    "{{ messages[0]['content'] + eos_token }}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{{ '{system_prompt}' + eos_token }}"
    "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ message['content'] }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] + eos_token }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"
    "{% endif %}"
)

# Replace with our specific template:
chat_template = chat_template.replace(
    "'{system_prompt}'", f"'{system_prompt}'"
).replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [ ]:

tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "What is 1+1?"},
        {
            "role": "assistant",
            "content": f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}",
        },
        {"role": "user", "content": "What is 2+2?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.
#
We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [ ]:

if not args.no_sft:
    print("[SFT] Pre-SFT formatting fine-tune enabled")
    from datasets import load_dataset
    import pandas as pd
    import numpy as np

    dataset = load_dataset(args.sft_data_path, split="cot")
    dataset = dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]

    # Try converting to number - if not, replace with NaN
    is_number = pd.to_numeric(
        pd.Series(dataset["expected_answer"]), errors="coerce"
    ).notnull()
    # Select only numbers
    dataset = dataset.iloc[np.where(is_number)[0]]

    dataset

    """We have to format the dataset to follow our GRPO style formatting:"""

    def format_dataset(x):
        expected_answer = x["expected_answer"]
        problem = x["problem"]

        # Remove generated think tags
        thoughts = x["generated_solution"]
        thoughts = thoughts.replace("<think>", "").replace("</think>", "")

        # Strip newlines on left and right
        thoughts = thoughts.strip()
        # Add our custom formatting
        final_prompt = (
            reasoning_start
            + thoughts
            + reasoning_end
            + solution_start
            + expected_answer
            + solution_end
        )
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": problem},
            {"role": "assistant", "content": final_prompt},
        ]

    dataset["Messages"] = dataset.apply(format_dataset, axis=1)

    """Check to see if it worked:"""

    tokenizer.apply_chat_template(dataset["Messages"][0], tokenize=False)

    """Let's truncate the pre fine-tuning dataset to `max_seq_length/2` since we don't want too long reasoning traces.

    Note this might take 2 minutes!
    """

    dataset["N"] = dataset["Messages"].apply(
        lambda x: len(tokenizer.apply_chat_template(x))
    )

    dataset = dataset.loc[dataset["N"] <= max_seq_length / 2].copy()
    dataset.shape

    """We then tokenize the messages and convert it to a Hugging Face compatible dataset format:"""

    from datasets import Dataset

    dataset["text"] = tokenizer.apply_chat_template(
        dataset["Messages"].values.tolist(), tokenize=False
    )
    dataset = Dataset.from_pandas(dataset)
    dataset

    """Let's now pre fine-tune the model so it follows our custom GRPO formatting!"""

    from trl import SFTTrainer, SFTConfig

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,  # Use GA to mimic batch size!
            warmup_steps=5,
            num_train_epochs=args.sft_epochs,  # Set this for 1 full training run.
            learning_rate=2e-4,  # Reduce to 2e-5 for long training runs
            logging_steps=5,
            optim=args.optim,
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=args.seed,
            report_to="wandb" if args.wandb else "none",  # WandB tracking
        ),
    )

    trainer.train()

    """Let's check if the model has learnt to follow the custom format:"""

    text = tokenizer.apply_chat_template(
        dataset[0]["Messages"][:2],
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
    )

    from transformers import TextStreamer

    _ = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        temperature=0,
        max_new_tokens=1024,
        streamer=TextStreamer(tokenizer, skip_prompt=False),
    )

    """Yes it did follow the formatting! Great! Let's remove some items before the GRPO step"""

    del dataset
    torch.cuda.empty_cache()
    import gc

    gc.collect()
else:
    print("[SFT] Skipping pre-SFT (--no-sft): GRPO-from-base comparison arm")

### Data Prep
<a name="Data"></a>
#
We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:

from datasets import load_dataset

dataset = load_dataset(args.grpo_data_path, "en", split="train")
if args.sample_n > 0:
    dataset = dataset.select(range(min(args.sample_n, len(dataset))))
    print(f"[data] Using subset of {len(dataset)} samples")
dataset

Let's look at the first row:

In [ ]:

dataset[0]["prompt"]

dataset[0]["solution"]

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:


def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text


extract_hash_answer(dataset[0]["solution"])

Let's map the dataset! and see the first row:

In [ ]:

dataset = dataset.map(
    lambda x: {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": x["prompt"]},
        ],
        "answer": extract_hash_answer(x["solution"]),
    }
)
dataset[0]

We create a regex format to match the reasoning sections and answers:

In [ ]:

import re

# Add optional EOS token matching
solution_end_regex = (
    r"</SOLUTION>[\s]{0,}" + "(?:" + re.escape(tokenizer.eos_token) + ")?"
)

match_format = re.compile(
    rf"{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end_regex}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)
match_format

We verify it works:

In [ ]:

match_format.findall(
    f"Let me think!<end_working_out><SOLUTION>\n2\n</SOLUTION>",
)

match_format.findall(
    f"<start_working_out>Let me think!<end_working_out><SOLUTION>  2  </SOLUTION>\n\n",
)

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:


def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:


def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward the opening tag since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        score += 0.5 if response.count(solution_start) == 1 else -1.0
        score += 0.5 if response.count(solution_end) == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:


def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if ratio >= 0.9 and ratio <= 1.1:
                    score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2:
                    score += 1.5
                else:
                    score -= 2.5  # Penalize wrong answers
            except:
                score -= 4.5  # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.
#
We also remove possible commas for example as in 123,456

In [ ]:

match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})", flags=re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:

global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5


def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            "*" * 20 + f"Question:\n{question}",
            f"\nAnswer:\n{answer[0]}",
            f"\nResponse:\n{responses[0]}",
            f"\nExtracted:\n{extracted_responses[0]}",
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!
#
Ie we'll remove the top 10% long prompts.

In [ ]:

tokenized = dataset.map(
    lambda x: {
        "tokens": tokenizer.apply_chat_template(
            x["prompt"], add_generation_prompt=True, tokenize=True
        )
    },
    batched=True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})

import numpy as np

maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

<a name="Train"></a>
### Train the model
#
Now set up GRPO Trainer and all configurations!

In [ ]:

max_prompt_length = maximum_length + 1  # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=args.seed,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=args.temperature,
    learning_rate=args.lr,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim=args.optim,
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # Increase to 4 for smoother training
    num_generations=args.num_generations,  # Ablation: group size
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps=args.max_steps,
    save_steps=args.max_steps,
    report_to="wandb" if args.wandb else "none",  # Weights & Biases
    output_dir=str(output_dir),
    seed=args.seed,
    run_name=RUN_NAME,
    # === Ablation knobs (research-backed) ===
    beta=args.beta,  # KL divergence penalty (0 => KL not computed/logged)
    loss_type=args.loss_type,  # grpo | bnpo | dr_grpo | dapo (normalization variants)
    importance_sampling_level=args.importance_sampling_level,  # token vs sequence advantage credit
    scale_rewards="group",  # group-normalized advantages (DeepSeekMath scheme)
    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!
#
You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!
#
| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:

# === WandB login (API key read from environment automatically) ===
if args.wandb:
    try:
        import wandb

        if os.environ.get("WANDB_API_KEY"):
            wandb.login(key=os.environ["WANDB_API_KEY"])
        print(
            f"[wandb] Logging to project={os.environ.get('WANDB_PROJECT')} "
            f"group={os.environ.get('WANDB_RUN_GROUP')} name={RUN_NAME} mode={os.environ.get('WANDB_MODE', 'online')}"
        )
    except Exception as e:
        print(f"[wandb] Init failed ({e}) — continuing without wandb")

# === Custom callback: dump EVERY logged metric per run ===
from transformers import TrainerCallback


class MetricsDumpCallback(TrainerCallback):
    """Serializes all trainer metrics to outputs/<run_name>/metrics.json.

    GRPOTrainer already logs (no duplication here): reward, reward_std,
    kl (only when beta != 0), entropy, completions/*, rewards/{func}/*,
    clip_ratio/*, loss, grad_norm, learning_rate, epoch.
    """

    def __init__(self, run_name, args_namespace, out_dir):
        self.run_name = run_name
        self.config = vars(args_namespace)
        self.out_dir = pathlib.Path(out_dir)
        self.metrics_history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        entry = {"step": state.global_step}
        entry.update({k: v for k, v in logs.items() if isinstance(v, (int, float))})
        self.metrics_history.append(entry)

    def on_train_end(self, args, state, control, **kwargs):
        dump = {
            "run_name": self.run_name,
            "config": self.config,
            "final_metrics": self.metrics_history[-1] if self.metrics_history else {},
            "metrics_history": self.metrics_history,
        }
        out_path = self.out_dir / "metrics.json"
        with open(out_path, "w") as f:
            json.dump(dump, f, indent=2, default=str)
        print(f"[metrics] Saved to {out_path}")


metrics_callback = MetricsDumpCallback(RUN_NAME, args, output_dir)

# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args=training_args,
    train_dataset=dataset,
    callbacks=[metrics_callback],
    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

# === Post-training metrics fallback (covers callback non-firing edge cases) ===
if not (output_dir / "metrics.json").exists():
    fallback_metrics = {
        "run_name": RUN_NAME,
        "config": vars(args),
        "final_metrics": trainer.state.log_history[-1]
        if trainer.state.log_history
        else {},
        "metrics_history": trainer.state.log_history,
    }
    with open(output_dir / "metrics.json", "w") as f:
        json.dump(fallback_metrics, f, indent=2, default=str)
    print(f"[metrics] Fallback dump saved to {output_dir / 'metrics.json'}")

if args.wandb:
    try:
        import wandb

        wandb.finish()
    except Exception:
        pass

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:

text = "What is the sqrt of 101?"

from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:

model.save_lora(str(output_dir / "grpo_saved_lora"))

Verify LoRA is actually trained!

In [ ]:

from safetensors import safe_open

tensors = {}
with safe_open(
    str(output_dir / "grpo_saved_lora" / "adapter_model.safetensors"), framework="pt"
) as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert n_zeros.item() != tensor.numel()

Now we load the LoRA and test:

In [ ]:

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,  # Must add for generation
    tokenize=False,
)
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=2048,
)
output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora(str(output_dir / "grpo_saved_lora")),
    )[0]
    .outputs[0]
    .text
)

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!
#
<a name="Save"></a>
### Saving to float16 for VLLM
#
We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:

# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
        token="YOUR_HF_TOKEN",
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
        token="YOUR_HF_TOKEN",
    )

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.
#
Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.
#
[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:

# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "qwen_finetune",
        tokenizer,
    )
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", tokenizer, token="YOUR_HF_TOKEN"
    )

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="f16")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="f16",
        token="YOUR_HF_TOKEN",
    )

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="q4_k_m",
        token="YOUR_HF_TOKEN",
    )

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",  # Change hf to your username!
        tokenizer,
        quantization_method=[
            "q4_k_m",
            "q8_0",
            "q5_k_m",
        ],
        token="YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.
#
And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!
#
Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!
#
<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>
#
  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
#
  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).